In [ ]:
# Environment info: useful for Colab/local reproducibility
import sys, platform, numpy as np
def where_am_i():
    try:
        import google.colab
        return "Google Colab"
    except ImportError:
        return "Local/other Jupyter"
print(f"Environment: {where_am_i()}")
print(f"Python: {sys.version.split()[0]}")
print(f"Numpy: {np.__version__}")

# Protein sequence alignment demo (global alignment)

## Main function (what you will reuse)
- `align_query_to_many(query, targets, ...)` aligns **one amino-acid query** against **many** targets and returns the top matches.
- Uses a simple global alignment (Needleman–Wunsch) with match/mismatch/gap scoring.

## Internals (what happens under the hood)
- Dynamic programming builds a **score matrix** (best score for every prefix).
- A **traceback** through pointers reconstructs the final aligned strings.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Iterable, Sequence, Literal, Any

import numpy as np

In [ ]:
@dataclass(frozen=True)
class AlignmentResult:
    target_id: str  # identifier for the target sequence
    score: int      # alignment score
    aligned_query: str  # aligned query string (with gaps)
    aligned_target: str # aligned target string (with gaps)

# Validate a protein sequence: checks type, emptiness, and allowed characters
def _validate_protein(seq: str, *, name: str = "sequence") -> str:
    if not isinstance(seq, str) or not seq:
        raise ValueError(f"{name} must be a non-empty string")
    seq = seq.strip().upper()  # remove whitespace, uppercase
    if not seq:
        raise ValueError(f"{name} must not be empty/whitespace")
    # Allow 20 AA + common ambiguity/stop symbols. Keep it permissive for teaching.
    allowed = set("ACDEFGHIKLMNPQRSTVWYBXZJUO*-" )
    bad = sorted({c for c in seq if c not in allowed})
    if bad:
        raise ValueError(f"{name} contains invalid characters: {bad}")
    return seq

# Needleman–Wunsch global alignment implementation
def needleman_wunsch(
    query: str,
    target: str,
    *,
    match: int = 1,
    mismatch: int = -1,
    gap: int = -2,
    tie_break: Sequence[Literal["diag", "up", "left"]] = ("diag", "up", "left"),
    return_matrices: bool = False,
) -> tuple[AlignmentResult, dict[str, Any] | None]:
    """Global alignment (Needleman–Wunsch) with simple match/mismatch/gap scoring.

    Returns:
      - AlignmentResult(score, aligned strings)
      - optional debug dict containing score matrix + traceback pointers
    """
    q = _validate_protein(query, name="query")  # validate query
    t = _validate_protein(target, name="target")  # validate target
    n, m = len(q), len(t)  # lengths of query and target

    # Initialize score and pointer matrices
    score = np.zeros((n + 1, m + 1), dtype=int)  # DP score matrix
    pointer = np.empty((n + 1, m + 1), dtype=object)  # traceback direction matrix

    # Set origin
    pointer[0, 0] = None
    # Initialize first column (gaps in target)
    for i in range(1, n + 1):
        score[i, 0] = score[i - 1, 0] + gap
        pointer[i, 0] = "up"
    # Initialize first row (gaps in query)
    for j in range(1, m + 1):
        score[0, j] = score[0, j - 1] + gap
        pointer[0, j] = "left"

    # Scoring function: +match, -mismatch
    def s(a: str, b: str) -> int:
        return match if a == b else mismatch

    # Fill DP matrix
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            diag = score[i - 1, j - 1] + s(q[i - 1], t[j - 1])  # match/mismatch
            up = score[i - 1, j] + gap  # gap in target
            left = score[i, j - 1] + gap  # gap in query
            candidates = {"diag": diag, "up": up, "left": left}
            best = max(candidates.values())  # best score
            # deterministic tie break for reproducibility
            for move in tie_break:
                if candidates[move] == best:
                    pointer[i, j] = move  # record move
                    break
            score[i, j] = best  # record score

    # Traceback: reconstruct alignment from bottom-right
    i, j = n, m
    aq: list[str] = []  # aligned query
    at: list[str] = []  # aligned target
    while i > 0 or j > 0:
        move = pointer[i, j]
        if move == "diag":
            aq.append(q[i - 1])
            at.append(t[j - 1])
            i -= 1
            j -= 1
        elif move == "up":
            aq.append(q[i - 1])
            at.append("-")
            i -= 1
        elif move == "left":
            aq.append("-")
            at.append(t[j - 1])
            j -= 1
        else:
            # Should only happen at (0,0)
            break

    aq_str = "".join(reversed(aq))  # final aligned query
    at_str = "".join(reversed(at))  # final aligned target

    result = AlignmentResult(
        target_id="",
        score=int(score[n, m]),
        aligned_query=aq_str,
        aligned_target=at_str,
    )
    debug = None
    if return_matrices:
        debug = {
            "query": q,
            "target": t,
            "score": score,
            "pointer": pointer,
            "params": {"match": match, "mismatch": mismatch, "gap": gap, "tie_break": list(tie_break)},
        }
    return result, debug

In [ ]:
# Align one query to many targets and return best matches
def align_query_to_many(
    query: str,
    targets: Sequence[str] | dict[str, str],
    *,
    match: int = 1,
    mismatch: int = -1,
    gap: int = -2,
    top_k: int = 5,
    return_alignments: bool = True,
    tie_break: Sequence[Literal["diag", "up", "left"]] = ("diag", "up", "left"),
) -> list[AlignmentResult]:
    """Align one query to many targets and return the best matches.

    `targets` can be:
      - list/tuple of sequences (ids will be 't0', 't1', ...)
      - dict of {id: sequence}
    """
    if top_k <= 0:
        raise ValueError("top_k must be >= 1")  # sanity check

    q = _validate_protein(query, name="query")  # validate query
    # Prepare (id, sequence) pairs
    if isinstance(targets, dict):
        items = list(targets.items())
    else:
        items = [(f"t{i}", s) for i, s in enumerate(targets)]

    results: list[AlignmentResult] = []
    for tid, tseq in items:
        # Align query to each target
        res, _ = needleman_wunsch(
            q,
            tseq,
            match=match,
            mismatch=mismatch,
            gap=gap,
            tie_break=tie_break,
            return_matrices=False,
        )
        # Optionally omit alignments for speed/memory
        if not return_alignments:
            res = AlignmentResult(target_id=tid, score=res.score, aligned_query="", aligned_target="")
        else:
            res = AlignmentResult(target_id=tid, score=res.score, aligned_query=res.aligned_query, aligned_target=res.aligned_target)
        results.append(res)

    # Sort by score, descending
    results.sort(key=lambda r: r.score, reverse=True)
    return results[: min(top_k, len(results))]  # return top_k

In [ ]:
# Quick usage: align one query against many targets
query = "MKTAYIAKQRQISFVKSHFS"
targets = {
    "t1": "MKTAYIAKQRQISFVKSHFA",
    "t2": "MKTAYIATRRQISFVKSHFS",
    "t3": "MKTAYIAKQRQISFVKSHLS",
    "t4": "MKTAYIAKQRQISFVKSHF",
    "t5": "MKTAYIAKQRQISFVKSHFS",
    "t6": "MKTTYIAKQGQISFVKSHYS",
    "t7": "MKTTYIAKQG",
    "t8": "MKTTYIAKISHFS",
}

hits = align_query_to_many(query, targets, match=2, mismatch=-1, gap=-2, top_k=15)
for h in hits:
    print(h.target_id, "score=", h.score)
    print(h.aligned_query)
    print(h.aligned_target)
    print()

In [ ]:
# Show how it works internally on ONE pair (score matrix + traceback)
q = "MKTWQ"
t = "MZTAQ"

res, dbg = needleman_wunsch(q, t, match=2, mismatch=-1, gap=-2, return_matrices=True)
print("Final score:", res.score)
print(res.aligned_query)
print(res.aligned_target)

score = dbg["score"]
pointer = dbg["pointer"]

print("\nScore matrix shape:", score.shape)
print(score)

In [ ]:
# Traceback path visualization (coordinates + moves)
def traceback_path(pointer: np.ndarray) -> list[tuple[int, int, str | None]]:
    i, j = pointer.shape[0] - 1, pointer.shape[1] - 1
    path: list[tuple[int, int, str | None]] = [(i, j, pointer[i, j])]
    while i > 0 or j > 0:
        move = pointer[i, j]
        if move == "diag":
            i -= 1; j -= 1
        elif move == "up":
            i -= 1
        elif move == "left":
            j -= 1
        else:
            break
        path.append((i, j, pointer[i, j]))
    return list(reversed(path))

path = traceback_path(pointer)
print("Path length:", len(path))
print("First 10 steps (i,j,move):")
for step in path[:10]:
    print(step)

In [ ]:
# Pretty-print the score matrix with row/col labels (small examples)
def format_score_matrix(score: np.ndarray, query: str, target: str) -> str:
    q = "-" + query
    t = "-" + target
    cell_w = max(3, max(len(str(int(x))) for x in score.flatten()) + 1)
    header = "".ljust(cell_w) + "".join(ch.rjust(cell_w) for ch in t)
    lines = [header]
    for i, ch in enumerate(q):
        row = ch.rjust(cell_w) + "".join(str(int(score[i, j])).rjust(cell_w) for j in range(len(t)))
        lines.append(row)
    return "\n".join(lines)

print(format_score_matrix(score, dbg["query"], dbg["target"]))

## How does the Needleman–Wunsch algorithm work?

The Needleman–Wunsch algorithm is a classic method for finding the best possible alignment between two sequences (like proteins or DNA). It guarantees the optimal global alignment, meaning it aligns the entire length of both sequences.

**How it works:**
1. **Scoring:** It uses a scoring system: + for matches, – for mismatches, and a penalty for gaps (insertions/deletions).
2. **Matrix filling:** It builds a grid (matrix) where each cell represents the best score for aligning the first part of sequence 1 to the first part of sequence 2. It fills this matrix from top-left to bottom-right, always choosing the best score at each step (match/mismatch/gap).
3. **Traceback:** Once the matrix is filled, it traces back from the bottom-right to the top-left, following the path of best scores. This path tells you exactly how to align the two sequences (where to put gaps, matches, mismatches).

**Why is it useful?**
- Finds the best overall alignment, not just local matches.
- Handles gaps and mismatches in a principled way.
- Used as the basis for many bioinformatics tools.

**In summary:**
Needleman–Wunsch is like a smart grid search that guarantees you find the best way to line up two sequences, even if they have differences or missing parts.